# Figure 3: Learning dynamics for different degrees of alignment

In [3]:
import numpy as np
import seaborn as sns
from networks.linear_network import LinearNetwork
from utils import get_random_regression_task, get_lambda_balanced, get_ntk, kernel_distance,BlindColours
from qqt.lambda_balanced_extra_dim_clean import QQT_lambda_balanced_unequal
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from qqt.lambda_balanced_extra_dim_clean import MTL_QQT_lambda_balanced_bottleneck
from qqt.lambda_balanced_extra_dim_bottel import QQT_lambda_balanced_unequal
from utils import get_random_regression_task, get_lambda_balanced,get_lambda_balanced_botteld, get_ntk, reshape_matrix, kernel_distance, cosine_similarity,BlindColours, zero_balanced_weights
import matplotlib as mpl
import matplotlib.patheffects as pe
import matplotlib.gridspec as gridspec
from string import ascii_lowercase as _abc
import torch
import torch.nn.functional as F
from matplotlib import ticker
from matplotlib.ticker import FormatStrFormatter
import random


## MTL Data Generation

In [4]:

from utils import whiten
from networks.linear_network import *
from utils import MTL_get_lambda_balanced_STLex
from qqt.lambda_balanced_extra_dim_clean import MTL_QQT_lambda_balanced_unequal


seed=5434
random.seed(seed)
np.random.seed(seed)
np.random.default_rng(seed=seed)

learning_rate = 0.00002
training_steps = 20000
in_dim=4
out_dim_1=1
out_dim_2=1
hidden_dim=2
batch_size=100



def get_random_regression_task_MTL(batch_size, in_dim, out_dim_1, out_dim_2):
    X = np.random.randn(batch_size, in_dim)
    Y1 = np.random.randn(batch_size, out_dim_1)
    Y2= np.random.randn(batch_size, out_dim_2)
    X_whitened = whiten(X)
    return X_whitened, Y1, Y2


def get_MTL_Tasks(batch_size, in_dim, out_dim_1, out_dim_2, alpha_scales=(1,1), alignment_factor=0.3, whitenli=True, seed=31):
    rng = np.random.default_rng(seed)

    X = np.random.randn(batch_size, in_dim)

    Q_full, _ = np.linalg.qr(rng.standard_normal((in_dim, in_dim)))
    W1=Q_full[:, :out_dim_1].T
    W2 = alignment_factor*W1 + (1-alignment_factor)*Q_full[:, out_dim_1:2*out_dim_1].T
    W2 = W2 / np.linalg.norm(W2, 'fro') * np.linalg.norm(W1, 'fro')
    W1= W1/ np.linalg.norm(W1, 'fro')*np.linalg.norm(W2,'fro')
    if whitenli == True:
        X = whiten(X)
    Y1 = alpha_scales[0]*W1 @ X.T
    Y2 = alpha_scales[1]*W2 @ X.T

    if out_dim_1 == 1:
      Y1=Y1.reshape(1, batch_size)
      Y2=Y2.reshape(1, batch_size)

    return X, Y1, Y2


def get_MTL_Tasks_spectrum(batch_size, in_dim, alphas, alpha_scales=(1,1),
                           whitenli=True, seed=None):
    """Two linear tasks whose overlap matrix O_12 has singular values exactly `alphas`.
       alphas[k] in [0,1]: 1 = shared mode, 0 = orthogonal, in-between = conflicting.
       Requires 2*len(alphas) <= in_dim  (i.e. N_o1 + N_o2 < N_1)."""
    rng = np.random.default_rng(seed)
    alphas = np.asarray(alphas, float)
    d = len(alphas)
    assert 2*d <= in_dim, f"need 2*{d} <= {in_dim} (N_o1+N_o2 < N_1)"
    assert np.all((alphas >= 0) & (alphas <= 1)), "alphas must be in [0,1]"

    X = np.random.randn(batch_size, in_dim)
    Q, _ = np.linalg.qr(rng.standard_normal((in_dim, in_dim)))
    A, B = Q[:, :d], Q[:, d:2*d]                       # mutually orthonormal blocks
    W1 = A.T
    W2 = alphas[:, None]*A.T + np.sqrt(1-alphas**2)[:, None]*B.T

    if whitenli:
        X = whiten(X)
    Y1 = alpha_scales[0]*W1 @ X.T
    Y2 = alpha_scales[1]*W2 @ X.T
    return X, Y1, Y2
    

## Training the networks

In [5]:


seeds = [0, 1, 2, 3, 4]
learning_rate = 0.0002
training_steps = 200000 #40000
in_dim=5
out_dim_1=1
out_dim_2=1
hidden_dim=2
batch_size=10
tasktype="MTL_ALI"
w1w1_empirical_mean = {}
w1w1_empirical_std = {}
w2w2_empirical_mean = {}
w2w2_empirical_std = {}
wsTws_empirical_mean = {}
wsTws_empirical_std = {}

Task1_losses_empirical_mean = {}
Task1_losses_empirical_std = {}
Task2_losses_empirical_mean = {}
Task2_losses_empirical_std = {}


cossim_empirical_mean = {}
cossim_empirical_std = {}

gradmag1_empirical_mean = {}
gradmag1_empirical_std = {}

gradmag2_empirical_mean = {}
gradmag2_empirical_std = {}

for lmda in ["MTL_ALI", "MTL_CONFLI", "MTL_Ortho"]:
    w1w1_runs = []
    w2w2_runs = []
    wsTws_runs = []
    task1_loss_runs = []
    task2_loss_runs = []
    cossim_runs = []
    gradmag1_runs = []
    gradmag2_runs = []
    for seed in seeds:
        random.seed(seed)
        np.random.seed(seed)

        if lmda == "MTL_ALI":
            X, Y1, Y2 = get_MTL_Tasks(
                batch_size, in_dim, out_dim_1, out_dim_2,
                alpha_scales=(1, 2), alignment_factor=1
            )
        elif lmda == "MTL_CONFLI":
            X, Y1, Y2 = get_MTL_Tasks(
                batch_size, in_dim, out_dim_1, out_dim_2,
                alpha_scales=(1, 2), alignment_factor=0.5
            )
        else:
            X, Y1, Y2 = get_MTL_Tasks(
                batch_size, in_dim, out_dim_1, out_dim_2,
                alpha_scales=(1, 2), alignment_factor=0
            )

        Y1 = Y1.T
        Y2 = Y2.T

        init_ws, init_w1, init_w2 = MTL_get_lambda_balanced_STLex(
            0, in_dim, hidden_dim, out_dim_1, out_dim_2, sigma_weights=0.01
        )

        MTLmodel = MTL_linearnetwork(
            in_dim, hidden_dim, out_dim_1, out_dim_2,
            init_ws=init_ws.copy(),
            init_w1=init_w1.copy(),
            init_w2=init_w2.copy()
        )

        wslist, w1list, w2list, losses1, losses2, cossim, gradmag1, gradmag2 = MTLmodel.trainMTL(
            X.T, Y1.T, Y2.T,
            training_steps, learning_rate,
            trackgradients=True,
            save_weight_history=True
        )

        w1w1_runs.append(np.array([w1 @ w1.T for w1 in w1list]))
        w2w2_runs.append(np.array([w2 @ w2.T for w2 in w2list]))
        wsTws_runs.append(np.array([ws.T @ ws for ws in wslist]))

        task1_loss_runs.append(np.array([
            1 / (2 * batch_size) * np.linalg.norm(w @ X.T - Y1.T) ** 2
            for w in [w1 @ ws for (w1, ws) in zip(w1list, wslist)]
        ]))

        task2_loss_runs.append(np.array([
            1 / (2 * batch_size) * np.linalg.norm(w @ X.T - Y2.T) ** 2
            for w in [w2 @ ws for (w2, ws) in zip(w2list, wslist)]
        ]))

        cossim_runs.append(np.asarray(cossim))
        gradmag1_runs.append(np.asarray(gradmag1))
        gradmag2_runs.append(np.asarray(gradmag2))

    w1w1_runs = np.asarray(w1w1_runs)
    w2w2_runs = np.asarray(w2w2_runs)
    wsTws_runs = np.asarray(wsTws_runs)
    task1_loss_runs = np.asarray(task1_loss_runs)
    task2_loss_runs = np.asarray(task2_loss_runs)

    w1w1_empirical_mean[lmda] = np.mean(w1w1_runs, axis=0)
    w1w1_empirical_std[lmda] = np.std(w1w1_runs, axis=0)

    w2w2_empirical_mean[lmda] = np.mean(w2w2_runs, axis=0)
    w2w2_empirical_std[lmda] = np.std(w2w2_runs, axis=0)

    wsTws_empirical_mean[lmda] = np.mean(wsTws_runs, axis=0)
    wsTws_empirical_std[lmda] = np.std(wsTws_runs, axis=0)

    Task1_losses_empirical_mean[lmda] = np.mean(task1_loss_runs, axis=0)
    Task1_losses_empirical_std[lmda] = np.std(task1_loss_runs, axis=0)

    Task2_losses_empirical_mean[lmda] = np.mean(task2_loss_runs, axis=0)
    Task2_losses_empirical_std[lmda] = np.std(task2_loss_runs, axis=0)

    cossim_runs = np.asarray(cossim_runs)
    gradmag1_runs = np.asarray(gradmag1_runs)
    gradmag2_runs = np.asarray(gradmag2_runs)

    cossim_empirical_mean[lmda] = np.mean(cossim_runs, axis=0)
    cossim_empirical_std[lmda] = np.std(cossim_runs, axis=0)

    gradmag1_empirical_mean[lmda] = np.mean(gradmag1_runs, axis=0)
    gradmag1_empirical_std[lmda] = np.std(gradmag1_runs, axis=0)

    gradmag2_empirical_mean[lmda] = np.mean(gradmag2_runs, axis=0)
    gradmag2_empirical_std[lmda] = np.std(gradmag2_runs, axis=0)

diffinitfunv [[8.67361738e-18 3.46944695e-18]
 [3.46944695e-18 1.56125113e-17]]


KeyboardInterrupt: 

In [ ]:
col_keys   = ["MTL_ALI", "MTL_CONFLI", "MTL_Ortho"]
row_titles = [r"$\alpha= 1$", r"$\alpha= 0.7$", r"$\alpha= 0$"]
metric_titles = ["Loss", "GradSim", "GradMag"]

cmap = plt.cm.get_cmap("viridis", 8)
fig, axs_ = plt.subplots(
    len(col_keys), 3,
    figsize=(8.0, 3.5 * len(col_keys) / 2),
    sharex="row",
    sharey="col"
)

zero_tol = 1e-3
task1color = cmap(4)
task2color = cmap(2)
gradsim_color = cmap(0)

plot_step = 120000
plot_step_ali = 120000
labelsize = 14
axessize = 12
band_alpha = 0.20

for j, title in enumerate(metric_titles):
    axs_[0, j].set_title(title, fontsize=labelsize, pad=15, fontweight="bold")

for i, lmda in enumerate(col_keys):
    axs = axs_[i, :]
    row_plot_step = plot_step_ali if lmda == "MTL_ALI" else plot_step

    task1_mean = np.asarray(Task1_losses_empirical_mean[lmda])[:row_plot_step]
    task1_std  = np.asarray(Task1_losses_empirical_std[lmda])[:row_plot_step]
    task2_mean = np.asarray(Task2_losses_empirical_mean[lmda])[:row_plot_step]
    task2_std  = np.asarray(Task2_losses_empirical_std[lmda])[:row_plot_step]
    x_loss = np.arange(len(task1_mean))

    zero_hits = np.where((task1_mean <= zero_tol) & (task2_mean <= zero_tol))[0]
    loss_zero_epoch = zero_hits[0] if len(zero_hits) else None

    axs[0].set_ylabel(row_titles[i], fontsize=labelsize, fontweight="bold", labelpad=18)

    axs[0].plot(x_loss, task1_mean, color=task1color)
    axs[0].fill_between(x_loss, task1_mean - task1_std, task1_mean + task1_std,
                        color=task1color, alpha=band_alpha)

    axs[0].plot(x_loss, task2_mean, color=task2color)
    axs[0].fill_between(x_loss, task2_mean - task2_std, task2_mean + task2_std,
                        color=task2color, alpha=band_alpha)

    axs[0].set_ylim(-0.1, 1.5)
    axs[0].set_yticks([0, 1.2])
    axs[0].tick_params(axis="y", labelsize=axessize)

    cossim_mean = np.asarray(cossim_empirical_mean[lmda])[:row_plot_step]
    cossim_std  = np.asarray(cossim_empirical_std[lmda])[:row_plot_step]
    x_cos = np.arange(len(cossim_mean))

    axs[1].plot(x_cos, cossim_mean, color=gradsim_color)
    axs[1].fill_between(x_cos, cossim_mean - cossim_std, cossim_mean + cossim_std,
                        color=gradsim_color, alpha=band_alpha)

    if loss_zero_epoch is not None:
        axs[1].axvline(loss_zero_epoch, color="k", linestyle="--", linewidth=1.2, alpha=0.7)

    axs[1].set_ylim(-1.1, 1.1)
    axs[1].tick_params(axis="y", labelsize=axessize)

    grad1_mean = np.asarray(gradmag1_empirical_mean[lmda])[:row_plot_step]
    grad1_std  = np.asarray(gradmag1_empirical_std[lmda])[:row_plot_step]
    grad2_mean = np.asarray(gradmag2_empirical_mean[lmda])[:row_plot_step]
    grad2_std  = np.asarray(gradmag2_empirical_std[lmda])[:row_plot_step]
    x_grad = np.arange(len(grad1_mean))

    axs[2].plot(x_grad, grad1_mean, color=task1color)
    axs[2].fill_between(x_grad, grad1_mean - grad1_std, grad1_mean + grad1_std,
                        color=task1color, alpha=band_alpha)

    axs[2].plot(x_grad, grad2_mean, color=task2color)
    axs[2].fill_between(x_grad, grad2_mean - grad2_std, grad2_mean + grad2_std,
                        color=task2color, alpha=band_alpha)

    axs[2].set_ylim(0, 0.85)
    axs[2].set_yticks([0, 0.5])
    axs[2].tick_params(axis="y", labelsize=axessize)

    for ax in axs:
        ax.set_xticks([0, row_plot_step])
        ax.tick_params(axis="x", labelbottom=True, labelsize=axessize)

letters = ["A", "B", "C"]
for j, letter in enumerate(letters):
    ax = axs_[0, j]
    ax.text(
        -0.15, 1.35, letter,
        transform=ax.transAxes,
        fontsize=16,
        fontweight="bold",
        va="top",
        ha="right"
    )

for ax in axs_[-1, :]:
    ax.set_xlabel("Epochs", fontsize=labelsize)
    ax.tick_params(axis="x", labelsize=axessize)

sns.despine()
fig.tight_layout()
plt.show()